# Treino do Modelo Transformer (DeBERTa v3 Small)


In [1]:

# !pip install transformers sentencepiece accelerate pandas scikit-learn

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score



c:\Users\joaop\miniconda3\envs\AP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carregamento e Preparação dos Dados

In [2]:

df_treino = pd.read_csv("../data/dataset_treino.csv")
df_val = pd.read_csv("../data/dataset_validacao.csv")

df_treino['Text'] = df_treino['Text'].fillna('')
df_val['Text'] = df_val['Text'].fillna('')

df_treino["source_name_limpo"] = df_treino["source_name"].astype(str).str.lower().str.strip()
df_val["source_name_limpo"] = df_val["source_name"].astype(str).str.lower().str.strip()

mapa_classes = {
    "human": 0,
    "openai": 1,
    "google": 2,
    "meta": 3,
    "anthropic": 4 
}

# Aplicar o mapeamento
df_treino["label_5_classes"] = df_treino["source_name_limpo"].map(mapa_classes)
df_val["label_5_classes"] = df_val["source_name_limpo"].map(mapa_classes)

# Remover quaisquer linhas que não tenham sido mapeadas corretamente (NaN)
df_treino = df_treino.dropna(subset=["label_5_classes"])
df_val = df_val.dropna(subset=["label_5_classes"])

# Converter para inteiros
df_treino["label_5_classes"] = df_treino["label_5_classes"].astype(int)
df_val["label_5_classes"] = df_val["label_5_classes"].astype(int)

print(f"Dados de Treino prontos: {len(df_treino)} exemplos.")
print(f"Dados de Validação prontos: {len(df_val)} exemplos.")

Dados de Treino prontos: 10880 exemplos.
Dados de Validação prontos: 2332 exemplos.


## 2. Preparação para o Hugging Face (Tokenização)
Os modelos Transformers não leem texto corrido. Precisamos de usar o `AutoTokenizer` do modelo específico (`microsoft/deberta-v3-small`) para transformar as frases em matrizes de números (IDs).

In [3]:
nome_modelo = "microsoft/deberta-v3-small"

print("A descarregar o Tokenizer da internet...")
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)

# Classe obrigatória para o PyTorch/Hugging Face gerir os dados em lotes (batches)
class AIDataset(Dataset):
    def __init__(self, textos, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            textos, 
            truncation=True, 
            padding=True, 
            max_length=max_length
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# Converter os nossos DataFrames para o formato Dataset do Hugging Face
train_dataset = AIDataset(df_treino['Text'].tolist(), df_treino['label_5_classes'].tolist(), tokenizer)
val_dataset = AIDataset(df_val['Text'].tolist(), df_val['label_5_classes'].tolist(), tokenizer)

print("Datasets convertidos para o formato PyTorch/Hugging Face com sucesso!")

A descarregar o Tokenizer da internet...


Datasets convertidos para o formato PyTorch/Hugging Face com sucesso!


## 3. Configuração do Modelo e Treino (Fine-Tuning)
Descarregar a arquitetura base do DeBERTa e adicionar-lhe uma "cabeça" de classificação para 5 classes. De seguida, configuramos o `Trainer` para gerir o ciclo de aprendizagem automaticamente.

In [ ]:
print("A descarregar a arquitetura do modelo...")

modelo = AutoModelForSequenceClassification.from_pretrained(nome_modelo, num_labels=5)

# Função para calcular a precisão (Accuracy) no final de cada época
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Argumentos do Treino (Hiperparâmetros)
args_treino = TrainingArguments(
    output_dir="./resultados_deberta",
    num_train_epochs=3,                   # Número de vezes que vê os dados todos
    per_device_train_batch_size=8,        # Se o PC ficar sem memória (OOM), muda para 4
    per_device_eval_batch_size=16,
    eval_strategy="epoch",                # Avalia na validação no fim de cada época
    save_strategy="epoch",                # Guarda checkpoints a cada época
    learning_rate=2e-5,                   # Taxa de aprendizagem baixa (recomendado p/ Transformers)
    weight_decay=0.01,                    # Regularização
    load_best_model_at_end=True,          # Guarda a versão que teve melhor resultado na validação
    metric_for_best_model="accuracy",
    report_to="none"                      # Desliga integrações externas (ex: wandb)
)

treinador = Trainer(
    model=modelo,
    args=args_treino,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("A iniciar o Fine-Tuning! (Pode demorar vários minutos/horas dependendo do hardware...)")
treinador.train()

A descarregar a arquitetura do modelo...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 17355.04it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier

A iniciar o Fine-Tuning! (Pode demorar vários minutos/horas dependendo do hardware...)


c:\Users\joaop\miniconda3\envs\AP\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
